# Xây Dựng Với Bộ Mô Hình Gia Đình Meta 

## Giới Thiệu 

Bài học này sẽ bao gồm: 

- Khám phá hai mô hình chính của gia đình Meta - Llama 3.1 và Llama 3.2, cùng các lựa chọn thay thế hiện có được sử dụng trong các ví dụ mã (Llama 3.3 và Llama 4) 
- Hiểu các trường hợp sử dụng và kịch bản cho từng mô hình 
- Ví dụ mã để thể hiện các tính năng riêng biệt của từng mô hình 


## Gia Đình Mô Hình Meta 

Trong bài học này, chúng ta sẽ khám phá 2 mô hình từ gia đình Meta hoặc "Llama Herd" - Llama 3.1 và Llama 3.2. Các ví dụ mã có thể chạy được sử dụng Llama 3.3 và Llama 4, đây là các lựa chọn thay thế hiện có trong danh mục Microsoft Foundry Models. 

Các mô hình này có các biến thể khác nhau và có sẵn trong [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Lưu ý:** GitHub Models sẽ ngừng hoạt động vào cuối tháng 7 năm 2026. Dưới đây là thêm chi tiết về việc sử dụng [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) để phát triển nguyên mẫu với các mô hình AI.

> **Lưu ý:** Các phiên bản mô hình cụ thể mà bài học này ban đầu dựa vào (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) đã được ngừng cung cấp trong danh mục Microsoft Foundry Models. Ví dụ mã bên dưới sử dụng `Llama-3.3-70B-Instruct` và `Llama-4-Scout-17B-16E-Instruct` như các lựa chọn thay thế hiện có trong danh mục Microsoft Foundry Models, nên vẫn hoạt động cho đến nay.

Các biến thể mô hình (như đã đề cập trong bài học này): 
- Llama 3.1 - 70B Instruct (đã ngừng) 
- Llama 3.1 - 405B Instruct (đã ngừng) 
- Llama 3.2 - 11B Vision Instruct (đã ngừng) 
- Llama 3.2 - 90B Vision Instruct (đã ngừng) 

Các lựa chọn thay thế hiện có được sử dụng trong ví dụ mã: 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Lưu ý: Các biến thể Llama 3 được thảo luận ở trên được giữ lại nhằm mục đích tham khảo lịch sử và hiện không còn có trong Microsoft Foundry Models.*



## Llama 3.1 

Với 405 Tỷ Tham Số, Llama 3.1 thuộc danh mục LLM mã nguồn mở. 

Phiên bản này là bản nâng cấp của Llama 3 trước đó với các cải tiến: 

- Cửa sổ ngữ cảnh lớn hơn - 128k token so với 8k token 
- Số lượng Token đầu ra tối đa lớn hơn - 4096 so với 2048 
- Hỗ trợ đa ngôn ngữ tốt hơn - nhờ tăng số token huấn luyện 

Những điều này cho phép Llama 3.1 xử lý các trường hợp phức tạp hơn khi xây dựng ứng dụng GenAI bao gồm: 
- Gọi hàm gốc - khả năng gọi công cụ và hàm ngoài ngoài quy trình LLM
- Hiệu suất RAG tốt hơn - nhờ cửa sổ ngữ cảnh lớn hơn 
- Tạo dữ liệu tổng hợp - khả năng tạo dữ liệu hiệu quả cho các tác vụ như tinh chỉnh 



### Gọi Hàm Gốc

Llama 3.1 đã được tinh chỉnh để hiệu quả hơn trong việc gọi hàm hoặc công cụ. Nó cũng có hai công cụ tích hợp mà mô hình có thể nhận biết cần sử dụng dựa trên lời nhắc từ người dùng. Các công cụ này là:

- **Brave Search** - Có thể được dùng để lấy thông tin cập nhật như thời tiết bằng cách thực hiện tìm kiếm trên web
- **Wolfram Alpha** - Có thể được dùng cho các phép tính toán học phức tạp hơn nên không cần phải viết hàm riêng.

Bạn cũng có thể tạo các công cụ tùy chỉnh cho LLM gọi.

Trong ví dụ mã dưới đây:

- Chúng ta định nghĩa các công cụ có sẵn (brave_search, wolfram_alpha) trong lời nhắc hệ thống.
- Gửi lời nhắc người dùng hỏi về thời tiết ở một thành phố cụ thể.
- LLM sẽ phản hồi với một gọi công cụ đến công cụ Brave Search trông giống như `<|python_tag|>brave_search.call(query="Stockholm weather")`

*Lưu ý: Ví dụ này chỉ thực hiện gọi công cụ, nếu bạn muốn có kết quả, bạn sẽ cần tạo tài khoản miễn phí trên trang API Brave và định nghĩa chức năng đó` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Mặc dù là một LLM, một hạn chế của Llama 3.1 là đa phương thức. Tức là, có thể sử dụng các loại đầu vào khác nhau như hình ảnh làm lời nhắc và cung cấp phản hồi. Khả năng này là một trong những tính năng chính của Llama 3.2. Các tính năng này cũng bao gồm:

- Đa phương thức - có khả năng đánh giá cả lời nhắc văn bản và hình ảnh 
- Các biến thể kích thước nhỏ đến trung bình (11B và 90B) - điều này cung cấp các tùy chọn triển khai linh hoạt, 
- Các biến thể chỉ văn bản (1B và 3B) - điều này cho phép mô hình được triển khai trên các thiết bị cận biên / di động và cung cấp độ trễ thấp 

Hỗ trợ đa phương thức đại diện cho một bước tiến lớn trong thế giới các mô hình nguồn mở. Ví dụ mã dưới đây sử dụng cả hình ảnh và lời nhắc văn bản để có được phân tích về hình ảnh. Vì Llama 3.2 90B Vision Instruct đã bị ngưng, ví dụ sử dụng `Llama-4-Scout-17B-16E-Instruct`, một lựa chọn thay thế đa phương thức bản địa hiện có. 

### Hỗ trợ đa phương thức với Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Việc học không dừng lại ở đây, hãy tiếp tục hành trình

Sau khi hoàn thành bài học này, hãy xem bộ sưu tập [Học tập AI Sinh tạo](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) của chúng tôi để tiếp tục nâng cao kiến thức về AI Sinh tạo của bạn!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
